# Exercice 1 : Détecter les renseignements personnels (PII) avec Presidio

> **Objectif.** Trouver automatiquement les renseignements personnels dans de vraies notes cliniques et de vrais journaux système.
> Mesurer ce que l'outil rate, puis le compléter avec nos propres détecteurs (MRN, adresse, nom d'utilisateur, adresse MAC, NAM québécois).

**Volets du plan de cours (séance 12) :** *préservation de la confidentialité des données* (trouver les renseignements personnels avant de les exposer) et *prévention des fuites d'information* (les PII qui traînent dans les journaux).

**Dépendances :** `presidio-analyzer`, `presidio-anonymizer`, modèle spaCy `en_core_web_lg`, `pandas` (installés par l'étape 0).

**Les données (dossier `data/`).**

| Fichier | Contenu | Provenance et licence |
|---|---|---|
| `notes_cliniques_synthea.json` | 40 notes cliniques (une par patient) | Générées par **Synthea** (MITRE), échantillon « latest » téléchargé de [synthea.mitre.org/downloads](https://synthea.mitre.org/downloads) (format FHIR R4). Licence **Apache 2.0**. Patients **synthétiques** : aucune personne réelle. On a ajouté à chaque note l'en-tête que produit un dossier médical électronique (nom, date de naissance, NAS, MRN, adresse, téléphone, médecin), tiré de la ressource FHIR `Patient` du même patient, et retiré les chiffres que Synthea colle aux noms (« Abraham100 »). |
| `OpenSSH_2k.log`, `Linux_2k.log`, `Thunderbird_2k.log` | 2 000 lignes de **vrais** journaux chacun : serveur SSH, système Linux, superordinateur Thunderbird (Sandia) | **Loghub** ([github.com/logpai/loghub](https://github.com/logpai/loghub)) : libre pour la recherche et l'enseignement, à condition de citer Zhu *et al.*, « Loghub », ISSRE 2023. Licence complète dans `data/LICENSE_loghub.txt`. |

Pourquoi Synthea et pas MIMIC ? Les notes de MIMIC-IV exigent une formation et une entente d'utilisation (DUA) sur PhysioNet, et la démo libre de MIMIC-IV ne contient **pas** de notes en texte libre.

**Le plan.**
1. Lire une note clinique et repérer les PII à l'œil.
2. Lancer Presidio tel quel et mesurer ce qu'il trouve sur les 40 notes.
3. Ajouter nos propres détecteurs et remesurer.
4. Remplacer les PII (dé-identification).
5. Passer aux journaux : adresses IP, noms d'utilisateur, adresses MAC.
6. Bonus québécois : le NAM, un identifiant qui contient des quasi-identifiants.

## Étape 0 : exécutez cette cellule en premier
Elle installe les paquets manquants et, sur Colab, récupère `helper.py` et `data/` depuis GitHub. Elle doit afficher **Prêt : True**.

In [1]:
# Dépendances de ce notebook : presidio-analyzer presidio-anonymizer
import os, sys, importlib.util
if not os.path.exists('helper.py'):   # notebook ouvert depuis Google Drive / Colab
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/8IAR805---Cybersecurity-AI.git /content/cours
    %cd "/content/cours/Chapitre 7 - IA et vie privée"
manquants = [pip for module, pip in {'presidio_analyzer': 'presidio-analyzer', 'presidio_anonymizer': 'presidio-anonymizer'}.items() if importlib.util.find_spec(module) is None]
if manquants:
    %pip install -q {' '.join(manquants)}
import spacy.util
if not spacy.util.is_package('en_core_web_lg'):
    !{sys.executable} -m spacy download en_core_web_lg -q
print('Prêt :', os.path.exists('helper.py') and os.path.exists('data'))

Prêt : True


## Étape 1 : Une vraie note clinique
**Ce qu'on fait :** on affiche le début d'une note, telle qu'elle sortirait du dossier médical électronique d'un hôpital.

In [2]:
from helper import charger_notes
notes = charger_notes()
print(len(notes), 'notes')
print(notes[0]['texte'][:1200])

40 notes
MELROSE FAMILY OPTICIANS LLC
PROGRESS NOTE
Patient: Abram Brekke    DOB: 1966-03-01    Sex: male
MRN: fd51467b-6db3-bca2-66fc-f19eeaaa7e33    SSN: 999-70-6180
Address: 755 Hilll Haven Suite 43, Medford, MA 02145    Phone: 555-643-7528
Attending: Dr. Elijah White

2024-06-18

# Chief Complaint
No complaints.

# History of Present Illness
Abram is a 58 year-old nonhispanic white male. Patient has a history of gingival disease (disorder), part-time employment (finding), medication review due (situation), full-time employment (finding), acute viral pharyngitis (disorder), victim of intimate partner abuse (finding), gingivitis (disorder), social isolation (finding), stress (finding).

# Social History
Patient is single. Patient quit smoking at age 20.
 Patient identifies as heterosexual.

Patient comes from a high socioeconomic background.
 Patient has a high school education.
Patient currently has UnitedHealthcare.

# Allergies
No Known Allergies.

# Medications
lisinopril 10 mg o

**👀 Ce qu'on doit voir :** un en-tête bourré d'**identifiants directs** (nom, NAS/SSN, MRN, téléphone, adresse), puis un récit clinique qui répète le prénom et contient des **quasi-identifiants** (âge, date de visite) et des renseignements de santé.

Sous HIPAA, presque tout cela fait partie des 18 identifiants à retirer (méthode *Safe Harbor*) : noms, adresses, **toutes les dates** sauf l'année, téléphones, numéros de dossier, etc.
Sous la Loi 25, tout renseignement qui permet d'identifier **directement ou indirectement** la personne est un renseignement personnel.

**Pour la suite**, on connaît la vérité terrain de chaque note (les vraies valeurs, venues du dossier structuré) :

In [3]:
notes[0]['pii']

{'PATIENT': 'Abram Brekke',
 'DATE_NAISSANCE': '1966-03-01',
 'SSN': '999-70-6180',
 'MRN': 'fd51467b-6db3-bca2-66fc-f19eeaaa7e33',
 'TELEPHONE': '555-643-7528',
 'ADRESSE': '755 Hilll Haven Suite 43',
 'VILLE': 'Medford',
 'CODE_POSTAL': '02145',
 'MEDECIN': 'Elijah White'}

## Étape 2 : Presidio, tel quel
**Ce qu'on fait :** Presidio (Microsoft, licence MIT) combine trois techniques : des **regex** (téléphone, NAS, IP…), un modèle de **NER** spaCy (personnes, lieux, dates) et des **mots de contexte** qui augmentent le score (« SSN: », « phone »).

In [4]:
from presidio_analyzer import AnalyzerEngine
import pandas as pd

analyseur = AnalyzerEngine()          # charge spaCy en_core_web_lg (~20 s la première fois)
texte = notes[0]['texte']
resultats = analyseur.analyze(texte, language='en')
pd.DataFrame([(r.entity_type, texte[r.start:r.end], round(r.score, 2)) for r in sorted(resultats, key=lambda r: r.start)],
             columns=['type', 'texte détecté', 'score']).head(20)

,type,texte détecté,score
0,PERSON,Abram Brekke,0.85
1,DATE_TIME,1966-03-01,0.85
2,US_SSN,999-70-6180,0.85
3,US_ITIN,999-70-6180,0.50
4,LOCATION,Medford,0.85
5,PERSON,MA,0.85
6,PHONE_NUMBER,555-643-7528,0.75
7,PERSON,Elijah White,0.85
8,DATE_TIME,2024-06-18\n\n#,0.85
9,DATE_TIME,58 year-old,0.85


**👀 Ce qu'on doit voir :** le nom, le NAS, le téléphone, les dates et le médecin sont trouvés. Mais regardez ce qui **manque** : le MRN, l'adresse de la rue, le code postal.
Et des fausses alertes : « MA » (l'État du Massachusetts) pris pour une **personne**, « reported.ph » pris pour une URL.

Une note, c'est une anecdote. **Mesurons sur les 40 notes** : pour chaque type de PII, quelle part des occurrences est entièrement masquée ?

In [5]:
from helper import evaluer_detection
avant = evaluer_detection(analyseur, notes)
avant

,occurrences,détectées,taux de détection
CODE_POSTAL,40,0,0.00
ADRESSE,40,1,0.02
MRN,40,4,0.10
PATIENT,40,32,0.80
VILLE,40,37,0.92
MEDECIN,40,39,0.98
SSN,40,40,1.00
DATE_NAISSANCE,40,40,1.00
TELEPHONE,40,40,1.00


**👀 Ce qu'on doit voir :** des types à 100 % (NAS, téléphone, date de naissance), le nom du patient raté dans 8 notes sur 40 (le NER rate les noms rares), et des types à 0 % ou presque : MRN (10 %), adresse (2 %), code postal (0 %).
Une note où il reste l'adresse et le code postal **n'est pas dé-identifiée** : c'est exactement la combinaison qu'utilise une attaque de liaison (exercice 3).

## Étape 3 : Ajouter nos propres détecteurs
**Ce qu'on fait :** un `PatternRecognizer` = une regex + un type + un score (+ des mots de contexte facultatifs).
Astuce de terrain : dans un document **structuré** (en-tête de dossier, formulaire), les étiquettes de champ (« Address: ») sont l'indice le plus fiable.

In [6]:
from presidio_analyzer import Pattern, PatternRecognizer

mrn = PatternRecognizer(supported_entity='MRN', context=['mrn', 'record'],
    patterns=[Pattern('uuid', r'\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b', 0.6)])
adresse = PatternRecognizer(supported_entity='ADRESSE',
    patterns=[Pattern('champ Address', r'(?<=Address: )[^,\n]+', 0.7)])
code_postal = PatternRecognizer(supported_entity='CODE_POSTAL',
    patterns=[Pattern('ZIP après état', r'(?<=\b[A-Z]{2} )\d{5}(?:-\d{4})?\b', 0.6)])

for r in [mrn, adresse, code_postal]:
    analyseur.registry.add_recognizer(r)

apres = evaluer_detection(analyseur, notes)
pd.concat({'avant': avant['taux de détection'], 'après': apres['taux de détection']}, axis=1).sort_values('avant')

,avant,après
CODE_POSTAL,0.00,1.00
ADRESSE,0.02,1.00
MRN,0.10,1.00
PATIENT,0.80,0.80
VILLE,0.92,0.92
MEDECIN,0.98,0.98
SSN,1.00,1.00
DATE_NAISSANCE,1.00,1.00
TELEPHONE,1.00,1.00


**👀 Ce qu'on doit voir :** MRN, adresse et code postal passent à 100 %. Les autres types ne bougent pas : le nom du patient reste à 80 %, il faudrait aussi un détecteur sur le champ « Patient: ».

**❓ Question pour la classe :** nos regex marchent parce que l'en-tête est toujours au même format. Que se passe-t-il si un médecin écrit l'adresse dans le récit libre (« patient lives at 12 Oak St ») ?

## Étape 4 : Dé-identifier
**Ce qu'on fait :** l'`AnonymizerEngine` remplace chaque détection. Ici, par son type entre chevrons (on pourrait aussi masquer, hacher ou chiffrer).

In [7]:
from presidio_anonymizer import AnonymizerEngine

anonymiseur = AnonymizerEngine()
texte = notes[0]['texte']
propre = anonymiseur.anonymize(text=texte, analyzer_results=analyseur.analyze(texte, language='en'))
print(propre.text[:900])

MELROSE FAMILY OPTICIANS LLC
PROGRESS NOTE
Patient: <PERSON>    DOB: <DATE_TIME>    Sex: male
MRN: <MRN>    SSN: <US_SSN>
Address: <ADRESSE>, <LOCATION>, <PERSON> <CODE_POSTAL>    Phone: <PHONE_NUMBER>
Attending: Dr. <PERSON>

<DATE_TIME> Chief Complaint
No complaints.

# History of Present Illness
Abram is a <DATE_TIME> nonhispanic white male. Patient has a history of gingival disease (disorder), part-time employment (finding), medication review due (situation), full-time employment (finding), acute viral pharyngitis (disorder), victim of intimate partner abuse (finding), gingivitis (disorder), social isolation (finding), stress (finding).

# Social History
Patient is single. Patient quit smoking at <DATE_TIME>.
 Patient identifies as heterosexual.

Patient comes from a high socioeconomic background.
 Patient has a high school education.
Patient currently has UnitedHealthcare.

# Allerg


**👀 Ce qu'on doit voir :** l'en-tête devient `<PERSON>`, `<US_SSN>`, `<MRN>`, `<ADRESSE>`… Le contenu clinique (diagnostics, médicaments) reste utilisable pour la recherche.
Notez aussi la **perte d'utilité** : l'âge (« 58 year-old ») a été remplacé alors qu'un chercheur en aurait besoin. HIPAA (*Safe Harbor*) autorise de garder l'âge, sauf au-delà de 89 ans (catégorie unique « 90 ans ou plus »).

## Étape 5 : Les journaux, côté cyber
**Ce qu'on fait :** même outil, sur un **vrai** journal SSH exposé à Internet. Les PII ici : adresses IP des clients, noms d'utilisateur, noms d'hôte.

In [8]:
from helper import charger_journal
ssh = charger_journal('OpenSSH_2k.log')
for ligne in ssh[:6]:
    print(ligne)
    print('   ->', [(r.entity_type, ligne[r.start:r.end]) for r in analyseur.analyze(ligne, language='en')])

Dec 10 06:55:46 LabSZ sshd[24200]: reverse mapping checking getaddrinfo for ns.marryaldkfaczcz.com [173.234.31.186] failed - POSSIBLE BREAK-IN ATTEMPT!
   -> [('DATE_TIME', 'Dec 10'), ('IP_ADDRESS', '173.234.31.186'), ('URL', 'ns.marryaldkfaczcz.com'), ('PHONE_NUMBER', '[173.234.31.186')]
Dec 10 06:55:46 LabSZ sshd[24200]: Invalid user webmaster from 173.234.31.186
   -> [('DATE_TIME', 'Dec 10'), ('IP_ADDRESS', '173.234.31.186'), ('PHONE_NUMBER', '173.234.31.186')]
Dec 10 06:55:46 LabSZ sshd[24200]: input_userauth_request: invalid user webmaster [preauth]
   -> [('DATE_TIME', 'Dec 10')]
Dec 10 06:55:46 LabSZ sshd[24200]: pam_unix(sshd:auth): check pass; user unknown
   -> [('DATE_TIME', 'Dec 10')]
Dec 10 06:55:46 LabSZ sshd[24200]: pam_unix(sshd:auth): authentication failure; logname= uid=0 euid=0 tty=ssh ruser= rhost=173.234.31.186 
   -> [('DATE_TIME', 'Dec 10'), ('IP_ADDRESS', '173.234.31.186'), ('PHONE_NUMBER', '173.234.31.186')]
Dec 10 06:55:48 LabSZ sshd[24200]: Failed password f

   -> [('DATE_TIME', 'Dec 10'), ('IP_ADDRESS', '173.234.31.186'), ('PHONE_NUMBER', '173.234.31.186')]


**👀 Ce qu'on doit voir :** les IP sont trouvées… mais aussi étiquetées `PHONE_NUMBER` (fausse alerte). Le nom d'utilisateur `webmaster` n'est **pas** trouvé : pour Presidio, ce n'est qu'un mot.

Deux corrections : (1) on dit à Presidio **quels types chercher** dans un journal (pas de téléphone ni de date ici), (2) on ajoute deux détecteurs, nom d'utilisateur et adresse MAC.

In [9]:
utilisateur = PatternRecognizer(supported_entity='UTILISATEUR', patterns=[Pattern('après un mot-clé',
    r'(?:(?<=[Ii]nvalid user )|(?<=Failed password for )|(?<=user=)|(?<=for user ))(?!invalid\b)[\w.\-]+', 0.7)])
mac = PatternRecognizer(supported_entity='MAC',
    patterns=[Pattern('MAC', r'\b(?:[0-9a-fA-F]{2}:){5}[0-9a-fA-F]{2}\b', 0.8)])
analyseur.registry.add_recognizer(utilisateur)
analyseur.registry.add_recognizer(mac)

TYPES_CYBER = ['IP_ADDRESS', 'UTILISATEUR', 'MAC', 'URL']

def pii_journal(lignes):
    trouvees = [(r.entity_type, l[r.start:r.end]) for l in lignes
                for r in analyseur.analyze(l, language='en', entities=TYPES_CYBER)]
    return pd.DataFrame(trouvees, columns=['type', 'valeur'])

for ligne in ssh[:6]:
    print([(r.entity_type, ligne[r.start:r.end]) for r in analyseur.analyze(ligne, language='en', entities=TYPES_CYBER)])

[('IP_ADDRESS', '173.234.31.186'), ('URL', 'ns.marryaldkfaczcz.com')]
[('UTILISATEUR', 'webmaster'), ('IP_ADDRESS', '173.234.31.186')]
[('UTILISATEUR', 'webmaster')]
[]
[('IP_ADDRESS', '173.234.31.186')]
[('UTILISATEUR', 'webmaster'), ('IP_ADDRESS', '173.234.31.186')]


**Ce qu'on fait :** on passe les trois journaux complets (6 000 lignes) et on compte les PII **distinctes** par type.

In [10]:
bilan = {}
for nom in ['OpenSSH_2k.log', 'Linux_2k.log', 'Thunderbird_2k.log']:
    pii = pii_journal(charger_journal(nom))
    bilan[nom] = pii.groupby('type')['valeur'].nunique()
pd.DataFrame(bilan).fillna(0).astype(int)

,OpenSSH_2k.log,Linux_2k.log,Thunderbird_2k.log
type,,,
IP_ADDRESS,30,68,17
MAC,0,0,1
URL,6,35,4
UTILISATEUR,63,5,1


In [11]:
pii_ssh = pii_journal(ssh)
pii_ssh[pii_ssh['type'] == 'UTILISATEUR']['valeur'].value_counts().head(10)

valeur
root       741
admin       87
support     18
oracle      18
test        15
user        12
uucp        10
0           10
inspur       9
1234         9
Name: count, dtype: int64

**👀 Ce qu'on doit voir :** des dizaines d'IP distinctes par journal (surtout des attaquants qui essaient des mots de passe), 63 noms d'utilisateur distincts dans le journal SSH (`root`, `admin`, `oracle`, `test`, et même `1234`), une adresse MAC dans les messages DHCP de Thunderbird, et des noms d'hôte (`URL`) qui contiennent parfois l'IP elle-même (`customer-187-141-143-180-sta.uninet-ide.com.mx`).

**❓ Question pour la classe :** l'IP d'un **attaquant** est-elle un renseignement personnel ? (Indice : une IP résidentielle identifie un abonné. Et l'IP de *vos* employés dans ces mêmes journaux ?)
Le Commissariat à la protection de la vie privée du Canada a conclu qu'une adresse IP est un renseignement personnel lorsqu'elle peut être associée à une personne identifiable, et la Cour suprême (*R. c. Bykovets*, 2024 CSC 6) y reconnaît une attente raisonnable de vie privée. Pour un SOC, la bonne réponse n'est pas d'effacer les IP, c'est de les **pseudonymiser** : c'est l'exercice 2.

## Étape 6 : Un identifiant québécois, le NAM
**Ce qu'on fait :** le numéro d'assurance maladie (NAM) de la RAMQ a 12 caractères. D'après sa description publique : 3 lettres du nom, 1 du prénom, année, mois (+50 pour une femme), jour, 2 chiffres.
Nos patients vivent au Massachusetts : on construit le NAM **fictif** qu'ils auraient au Québec.

In [12]:
from helper import nam_quebec
n = notes[1]
prenom, nom = n['pii']['PATIENT'].split(' ', 1)
nam = nam_quebec(prenom, nom, n['pii']['DATE_NAISSANCE'], n['sexe'])
phrase = f"Patient {n['pii']['PATIENT']}, NAM {nam}, vu à l'urgence."
print(phrase)

regex_nam = r'\b[A-Z]{4}\d{2}(?:0[1-9]|1[0-2]|5[1-9]|6[0-2])(?:0[1-9]|[12]\d|3[01])\d{2}\b'
nam_rec = PatternRecognizer(supported_entity='NAM_QC', context=['nam', 'ramq', 'assurance'],
                            patterns=[Pattern('NAM', regex_nam, 0.5)])
analyseur.registry.add_recognizer(nam_rec)
[(r.entity_type, phrase[r.start:r.end], round(r.score, 2)) for r in analyseur.analyze(phrase, language='en', entities=['NAM_QC'])]

Patient Albina Pagac, NAM PAGA12582201, vu à l'urgence.


[('NAM_QC', 'PAGA12582201', 0.85)]

**Le piège :** même « masqué » partiellement, un NAM **contient** des quasi-identifiants. Décodons-le :

In [13]:
def decoder_nam(nam):
    aa, mm, jj = int(nam[4:6]), int(nam[6:8]), int(nam[8:10])
    return {'début du nom': nam[:3], 'initiale': nam[3], 'sexe': 'F' if mm > 50 else 'M',
            'naissance': f'{aa:02d}-{mm - 50 if mm > 50 else mm:02d}-{jj:02d}'}
decoder_nam(nam)

{'début du nom': 'PAG', 'initiale': 'A', 'sexe': 'F', 'naissance': '12-08-22'}

**👀 Ce qu'on doit voir :** le sexe et la date de naissance complète (AA-MM-JJ). Hacher le NAM ne règle rien si l'attaquant peut énumérer les combinaisons possibles (quelques millions seulement).

## Ce qu'il faut retenir
- Une PII, c'est un **identifiant direct** (nom, NAS, MRN, IP, nom d'utilisateur, MAC) ou un **quasi-identifiant** (date de naissance, code postal, sexe) qui identifie en combinaison.
- Presidio donne une bonne base (regex + NER + contexte), mais **il faut mesurer** sur ses propres données : ici, MRN, adresse et code postal passaient à travers.
- Les détecteurs personnalisés (regex de champ, NAM, utilisateur, MAC) et le choix des types à chercher selon la source font la différence.
- Dé-identifier coûte de l'utilité (l'âge disparaît). Pour les IP d'un SOC, on ne veut pas effacer : on veut **pseudonymiser**.

**Exercice suivant :** pseudonymiser les IP et les horodatages de UNSW-NB15 sans casser la détection d'anomalies.